In [47]:
import pandas as pd
import numpy as np

df = pd.read_csv("Churn_Modelling.csv")

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [48]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

print(df.shape)
print(df.columns.tolist())

(10000, 12)
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain']


In [49]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

split_list = [0.4, 0.3, 0.2]
seed_list = [42, 7]

test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(8000, 11) (2000, 11)
Exited
0    6370
1    1630
Name: count, dtype: int64
Exited
0    1593
1     407
Name: count, dtype: int64


In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'SVM': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

rows = []

for name, model in models.items():
    if name in ['Logistic Regression', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rows.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        confusion_matrix(y_test, y_pred)
    ])

result_df = pd.DataFrame(rows, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Confusion Matrix'])
print(result_df)

                 Model  Accuracy  Precision    Recall        F1  \
0  Logistic Regression    0.8080   0.589147  0.186732  0.283582   
1                  SVM    0.8610   0.834197  0.395577  0.536667   
2        Decision Tree    0.7865   0.477578  0.523342  0.499414   
3        Random Forest    0.8610   0.769874  0.452088  0.569659   

            Confusion Matrix  
0    [[1540, 53], [331, 76]]  
1   [[1561, 32], [246, 161]]  
2  [[1360, 233], [194, 213]]  
3   [[1538, 55], [223, 184]]  


In [51]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

split_list = [0.4, 0.3, 0.2]
seed_list = [42, 7]
k_list = [3, 5, 10]

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ])
}

rows = []

for split in split_list:
    for seed in seed_list:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=split, random_state=seed, stratify=y
        )

        for name, model in models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            rows.append([
                name,
                split,
                seed,
                accuracy_score(y_test, y_pred),
                precision_score(y_test, y_pred, zero_division=0),
                recall_score(y_test, y_pred, zero_division=0),
                f1_score(y_test, y_pred, zero_division=0)
            ])

result_split_seed = pd.DataFrame(
    rows,
    columns=['Model', 'Test Size', 'Seed', 'Accuracy', 'Precision', 'Recall', 'F1']
)

print(result_split_seed)

                  Model  Test Size  Seed  Accuracy  Precision    Recall  \
0   Logistic Regression        0.4    42  0.813000   0.620072  0.212270   
1                   SVM        0.4    42  0.859500   0.821883  0.396319   
2         Decision Tree        0.4    42  0.780000   0.463687  0.509202   
3         Random Forest        0.4    42  0.863250   0.756705  0.484663   
4   Logistic Regression        0.4     7  0.809250   0.584416  0.220859   
5                   SVM        0.4     7  0.854000   0.789474  0.386503   
6         Decision Tree        0.4     7  0.788250   0.481651  0.515337   
7         Random Forest        0.4     7  0.860500   0.763860  0.456442   
8   Logistic Regression        0.3    42  0.812667   0.628272  0.196399   
9                   SVM        0.3    42  0.860667   0.836237  0.392799   
10        Decision Tree        0.3    42  0.797667   0.503185  0.517185   
11        Random Forest        0.3    42  0.866667   0.772610  0.489362   
12  Logistic Regression  

In [52]:
cv_rows = []

for k in k_list:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

        cv_rows.append([
            name,
            k,
            scores.mean(),
            scores.std()
        ])

result_cv = pd.DataFrame(
    cv_rows,
    columns=['Model', 'KFold', 'Mean CV Accuracy', 'Std']
)

print(result_cv)

                  Model  KFold  Mean CV Accuracy       Std
0   Logistic Regression      3            0.8107  0.002974
1                   SVM      3            0.8548  0.001702
2         Decision Tree      3            0.7917  0.003403
3         Random Forest      3            0.8568  0.003321
4   Logistic Regression      5            0.8107  0.005492
5                   SVM      5            0.8563  0.004202
6         Decision Tree      5            0.7937  0.002926
7         Random Forest      5            0.8623  0.005555
8   Logistic Regression     10            0.8098  0.006969
9                   SVM     10            0.8574  0.005553
10        Decision Tree     10            0.7904  0.007392
11        Random Forest     10            0.8608  0.011205


In [53]:
best_split_seed = result_split_seed.sort_values(by='Accuracy', ascending=False)
best_cv = result_cv.sort_values(by='Mean CV Accuracy', ascending=False)

print(best_split_seed.head(10))
print(best_cv)

            Model  Test Size  Seed  Accuracy  Precision    Recall        F1
23  Random Forest        0.2     7  0.874000   0.794677  0.513514  0.623881
21            SVM        0.2     7  0.870000   0.845070  0.442260  0.580645
15  Random Forest        0.3     7  0.869333   0.792000  0.486088  0.602434
11  Random Forest        0.3    42  0.866667   0.772610  0.489362  0.599198
13            SVM        0.3     7  0.863333   0.831683  0.412439  0.551422
3   Random Forest        0.4    42  0.863250   0.756705  0.484663  0.590875
17            SVM        0.2    42  0.861000   0.834197  0.395577  0.536667
19  Random Forest        0.2    42  0.861000   0.769874  0.452088  0.569659
9             SVM        0.3    42  0.860667   0.836237  0.392799  0.534521
7   Random Forest        0.4     7  0.860500   0.763860  0.456442  0.571429
                  Model  KFold  Mean CV Accuracy       Std
7         Random Forest      5            0.8623  0.005555
11        Random Forest     10            0.86

In [54]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

y_pred = best_rf.predict(X_test)

print("Best Params:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy: 0.8735
Precision: 0.7873134328358209
Recall: 0.5184275184275184
F1: 0.6251851851851852
Confusion Matrix:
 [[1536   57]
 [ 196  211]]


In [55]:
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, random_state=42))
])

svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, random_state=42))
])

rf = RandomForestClassifier(random_state=42)

voting = VotingClassifier(
    estimators=[('lr', lr), ('svm', svm), ('rf', rf)],
    voting='soft'
)

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)

print("Voting Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Voting Accuracy: 0.868
Precision: 0.8235294117647058
Recall: 0.44717444717444715
F1: 0.5796178343949044
Confusion Matrix:
 [[1554   39]
 [ 225  182]]


In [56]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X)

sil = silhouette_score(X, clusters)
ari = adjusted_rand_score(y, clusters)

print("Silhouette Score:", sil)
print("ARI:", ari)

Silhouette Score: 0.46733949558209104
ARI: -0.018066756877637154


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

split_rows = []

lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, random_state=42))
])

for mode in ['Stratified', 'Non-Stratified']:
    if mode == 'Stratified':
        X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    else:
        X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

    lr_model.fit(X_train_s, y_train_s)
    y_pred_s = lr_model.predict(X_test_s)

    split_rows.append([
        mode,
        accuracy_score(y_test_s, y_pred_s),
        precision_score(y_test_s, y_pred_s, zero_division=0),
        recall_score(y_test_s, y_pred_s, zero_division=0),
        f1_score(y_test_s, y_pred_s, zero_division=0),
        confusion_matrix(y_test_s, y_pred_s)
    ])

split_compare_df = pd.DataFrame(
    split_rows,
    columns=['Split Type', 'Accuracy', 'Precision', 'Recall', 'F1', 'Confusion Matrix']
)

print(split_compare_df)

       Split Type  Accuracy  Precision    Recall        F1  \
0      Stratified     0.808   0.589147  0.186732  0.283582   
1  Non-Stratified     0.811   0.552448  0.201018  0.294776   

          Confusion Matrix  
0  [[1540, 53], [331, 76]]  
1  [[1543, 64], [314, 79]]  


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

svm_basic = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])

svm_basic.fit(X_train, y_train)
y_pred_basic = svm_basic.predict(X_test)

print("Basic SVM")
print("Accuracy:", accuracy_score(y_test, y_pred_basic))
print("Precision:", precision_score(y_test, y_pred_basic, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_basic, zero_division=0))
print("F1:", f1_score(y_test, y_pred_basic, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_basic))

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])

param_grid_svm = {
    'svm__kernel': ['rbf', 'linear'],
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto']
}

cv_svm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_svm = GridSearchCV(
    svm_pipe,
    param_grid_svm,
    cv=cv_svm,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
y_pred_tuned = best_svm.predict(X_test)

print("\nTuned SVM")
print("Best Params:", grid_svm.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_tuned, zero_division=0))
print("F1:", f1_score(y_test, y_pred_tuned, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tuned))

Basic SVM
Accuracy: 0.87
Precision: 0.8450704225352113
Recall: 0.44226044226044225
F1: 0.5806451612903226
Confusion Matrix:
 [[1560   33]
 [ 227  180]]
